In [1]:
import pandas as pd
import requests

# 📥 Load your spreadsheet
df = pd.read_excel("AllVariantsCFTR-France_03-09-2024.xlsx")

# 🔑 Key columns
legacy_col = "Legacy name"
hgvs_col = "HGVS name (NM_000492.3)"
dna_type_col = "DNA type"
prot_type_col = "type_prot"

# 🎯 Known legacy-based class map
legacy_class_map = {
    "F508del": "Class II", "I507del": "Class II",
    "G551D": "Class III", "S549N": "Class III", "S549R": "Class III",
    "R334W": "Class IV", "R117H": "Class IV",
    "A455E": "Class V", "3849+10kbC>T": "Class V",
    "G542X": "Class I", "W1282X": "Class I", "R553X": "Class I",
    "621+1G>T": "Class I", "1811+1.6kbA>G": "Class I", "3120+1G>A": "Class I",
    "CFTRdele2,3": "Class I"
}

# 🌐 ClinVar API query
def query_clinvar_class(hgvs_name):
    url = f"https://www.ncbi.nlm.nih.gov/clinvar/?term={hgvs_name}"
    try:
        response = requests.get(url, timeout=5)
        if "pathogenic" in response.text.lower():
            return "Class I"
        elif "likely pathogenic" in response.text.lower():
            return "Class I"
        elif "uncertain significance" in response.text.lower():
            return "Unclassified"
        elif "benign" in response.text.lower():
            return "Unclassified"
    except:
        return None
    return None

# 🧠 Heuristic fallback
def infer_class(row):
    legacy = str(row[legacy_col]).strip()
    hgvs = str(row[hgvs_col]).strip()
    dna_type = str(row[dna_type_col]).lower()
    prot_type = str(row[prot_type_col]).lower()

    # 1. Legacy match
    if legacy in legacy_class_map:
        return legacy_class_map[legacy]

    # 2. ClinVar query
    clinvar_class = query_clinvar_class(hgvs)
    if clinvar_class:
        return clinvar_class

    # 3. Heuristic rules
    if "stop codon" in prot_type or "frameshift" in dna_type:
        return "Class I"
    if "del" in dna_type and "misfold" in prot_type:
        return "Class II"
    if "missense" in dna_type:
        return "Unclassified missense"
    return "Unclassified"

# 🧬 Apply classification
df["Functional Class"] = df.apply(infer_class, axis=1)

# 💾 Save results
df.to_csv("cftr_classified_with_clinvar.csv", index=False)
print("✅ All variants classified and saved.")


✅ All variants classified and saved.


In [9]:
import pandas as pd
import json

# ------------------------------------
# 1. Load France Variant Excel
# ------------------------------------
df = pd.read_excel("AllVariantsCFTR-France_03-09-2024.xlsx", sheet_name=0)

# ------------------------------------
# 2. Define CFTR Class Mapping Logic
# ------------------------------------
def infer_cftr_class(row):
    prot_type = str(row.get("type_prot", "")).lower()
    dna_type = str(row.get("DNA type", "")).lower()

    if "stop" in prot_type or "frameshift" in prot_type or "nonsense" in dna_type:
        return "I"
    if "misfold" in prot_type or "inframe" in prot_type:
        return "II"
    if "gating" in prot_type or "gating" in dna_type:
        return "III"
    if "conductance" in prot_type or "conductance" in dna_type:
        return "IV"
    if "reduced" in dna_type or "splice" in dna_type:
        return "V"
    return None  # Could not infer class

# ------------------------------------
# 3. Apply Inference and Clean
# ------------------------------------
df["cftr_class"] = df.apply(infer_cftr_class, axis=1)

# Optional: preview distribution
print("✅ Inferred class distribution:")
print(df["cftr_class"].value_counts(dropna=False))

# ------------------------------------
# 4. Export Class Labels for Known Variants
# ------------------------------------
variant_class_map = {}

for _, row in df.iterrows():
    legacy_name = str(row.get("Legacy name", "")).strip()
    cftr_class = row.get("cftr_class", None)

    if legacy_name and cftr_class:
        variant_class_map[legacy_name] = {"cftr_class": cftr_class}

# ------------------------------------
# 5. Save as JSON for Modeling
# ------------------------------------
with open("cftr2_variant_classes.json", "w") as f:
    json.dump(variant_class_map, f, indent=2)

print(f"✅ Saved {len(variant_class_map)} class-labeled variants to cftr2_variant_classes.json")


✅ Inferred class distribution:
cftr_class
None    895
I       215
II       10
Name: count, dtype: int64
✅ Saved 198 class-labeled variants to cftr2_variant_classes.json


In [10]:
import pandas as pd
import json

# -----------------------------
# 1. Load Confirmed Variant List
# -----------------------------
confirmed = pd.read_csv("variant_url_log.csv", names=["Legacy name", "URL", "Status"])
confirmed_variants = confirmed["Legacy name"].dropna().unique().tolist()

# -----------------------------
# 2. Load and Combine Excel Files
# -----------------------------
df1 = pd.read_excel("Book1.xlsx", sheet_name=0)
df2 = pd.read_excel("Book3.xlsx", sheet_name=0)
df_all = pd.concat([df1, df2], ignore_index=True)

# -----------------------------
# 3. Filter to Confirmed Variants
# -----------------------------
df_filtered = df_all[df_all["Protein change"].isin(confirmed_variants)]

# -----------------------------
# 4. Define CFTR Class Inference Logic
# -----------------------------
def infer_cftr_class(consequence):
    consequence = str(consequence).lower()
    if "frameshift" in consequence or "stop gained" in consequence or "nonsense" in consequence:
        return "I"
    if "inframe deletion" in consequence or "deletion" in consequence:
        return "II"
    if "missense" in consequence:
        return "III"  # Default → can adjust if known to be Class IV
    if "splice" in consequence or "intron" in consequence:
        return "V"
    if "start lost" in consequence or "stop lost" in consequence:
        return "I"
    return None

# -----------------------------
# 5. Apply and Clean Results
# -----------------------------
df_filtered["cftr_class"] = df_filtered["Molecular consequence"].apply(infer_cftr_class)
df_classmap = df_filtered[["Protein change", "cftr_class"]].dropna()

variant_classes = {
    str(row["Protein change"]).strip(): {"cftr_class": row["cftr_class"]}
    for _, row in df_classmap.iterrows()
}

# -----------------------------
# 6. Save Class Labels
# -----------------------------
with open("cftr2_variant_classes.json", "w") as f:
    json.dump(variant_classes, f, indent=2)

print(f"✅ Saved {len(variant_classes)} class-labeled variants to cftr2_variant_classes.json")


✅ Saved 6 class-labeled variants to cftr2_variant_classes.json


C:\Users\adith\AppData\Local\Temp\ipykernel_1916\3521032655.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered["cftr_class"] = df_filtered["Molecular consequence"].apply(infer_cftr_class)
